# Checkpoint VI · Examen blanc — solutions (notebook exécuté)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cemah2/workbookIA/blob/main/checkpoints/partie_6/03_examen_solutions.ipynb)

Les solutions des deux questions de code (CP6.8, la loss d'un VAE ; CP6.9, le pas de GAN réparé et entraîné), exécutées, puis l'enregistrement des réponses que vérifie la partie B du notebook de l'examen (cellules marquées `answer`, récoltées par `tools/build_answers.py`). Le corrigé détaillé et le barème sont dans `03_examen_corrige.md`.

In [ ]:
# ⚙️ SETUP: run this cell first (on Colab or on your computer)
FAST_MODE = True  # True = quick version (CPU, a few minutes); False = full version
SEED = 42

import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/cemah2/workbookIA.git"
DRIVE_DIR = Path("/content/drive/MyDrive/workbookIA")

try:
    import google.colab  # noqa: F401  (this module only exists on Colab)
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

if IN_COLAB:
    from google.colab import drive

    drive.mount("/content/drive")
    subprocess.run(["git", "config", "--global", "--add", "safe.directory", str(DRIVE_DIR)])
    if not (DRIVE_DIR / ".git").exists():
        subprocess.run(["git", "clone", REPO_URL, str(DRIVE_DIR)], check=True)
    else:
        # Get the new chapters. Your own work only lives in mon_travail/, which Claude never
        # touches, so "rebase + autostash" keeps it safe (even with local commits).
        identity = subprocess.run(["git", "-C", str(DRIVE_DIR), "config", "user.email"],
                                  capture_output=True).returncode == 0
        who = [] if identity else ["-c", "user.name=Workbook", "-c", "user.email=workbook@localhost"]
        pull = subprocess.run(["git", *who, "-C", str(DRIVE_DIR), "pull", "--rebase", "--autostash"],
                              capture_output=True, text=True)
        conflicts = subprocess.run(["git", "-C", str(DRIVE_DIR), "diff", "--name-only", "--diff-filter=U"],
                                   capture_output=True, text=True).stdout.split()
        if conflicts:  # a file changed outside mon_travail/ AND by Claude: git leaves conflict markers in it
            print("⚠️ Tes modifications de " + ", ".join(conflicts) + " (hors de mon_travail/) entrent en conflit "
                  "avec la nouvelle version ; elles sont gardées dans `git stash`. Pour reprendre la version du "
                  "dépôt : git restore --source=HEAD --staged --worktree <fichier> (00_setup/COLAB.md, « Problèmes "
                  "fréquents »).")
        elif pull.returncode != 0:
            print("⚠️ git pull a échoué (voir 00_setup/COLAB.md, « Problèmes fréquents ») :\n" + pull.stderr)
    ROOT = DRIVE_DIR
else:
    ROOT = Path(os.environ.get("WB_ROOT", Path.cwd())).resolve()
    while not (ROOT / "src" / "wb").exists() and ROOT != ROOT.parent:
        ROOT = ROOT.parent
    if not (ROOT / "src" / "wb").exists():
        raise FileNotFoundError("Dépôt introuvable : ouvre ce notebook depuis le dossier du workbook.")

os.environ["WB_ROOT"] = str(ROOT)
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import wb  # noqa: E402

cfg = wb.setup(seed=SEED, fast=FAST_MODE)
FAST_MODE = cfg.fast

## Partie A · Les questions de code

In [ ]:
# Data and tools of the code questions CP6.8 and CP6.9 (run this cell first)
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

# CP6.8: a batch of 16 MNIST images in [0, 1] (N, 1, 28, 28), the logits a decoder gave for them, and the two heads
# of an encoder with d = 4 latents (mu, log_var); everything in float64
_X8, _ = wb.datasets.load_mnist("train", n=16, normalize=True, seed=68)
X_V = torch.from_numpy(_X8.astype(np.float64)).unsqueeze(1)                        # (16, 1, 28, 28)
_g8 = torch.Generator().manual_seed(68)
LOGITS_V = 2.0 * torch.randn(16, 1, 28, 28, generator=_g8, dtype=torch.float64)  # (16, 1, 28, 28)
MU_V = torch.randn(16, 4, generator=_g8, dtype=torch.float64)                     # (16, 4)
LOGVAR_V = 0.5 * torch.randn(16, 4, generator=_g8, dtype=torch.float64)           # (16, 4)
_ORIGINALS = {"X_V": X_V.clone(), "LOGITS_V": LOGITS_V.clone(), "MU_V": MU_V.clone(), "LOGVAR_V": LOGVAR_V.clone()}

# CP6.9: a 1D GAN; the data follow N(4, 1.25^2)
torch.set_num_threads(1)                                                          # tiny networks: one thread is fastest
bce = nn.BCEWithLogitsLoss()                                                      # the mean over the mini-batch


def make_gan(seed=69):
    """(G, D, opt_d, opt_g): a new generator 1-16-1 and discriminator 1-32-32-1 (it gives a logit), Adam(2e-3)."""
    torch.manual_seed(seed)
    G = nn.Sequential(nn.Linear(1, 16), nn.Tanh(), nn.Linear(16, 1))
    D = nn.Sequential(nn.Linear(1, 32), nn.LeakyReLU(0.2), nn.Linear(32, 32), nn.LeakyReLU(0.2), nn.Linear(32, 1))
    opt_d = torch.optim.Adam(D.parameters(), lr=2e-3, betas=(0.5, 0.999))
    opt_g = torch.optim.Adam(G.parameters(), lr=2e-3, betas=(0.5, 0.999))
    return G, D, opt_d, opt_g


def real_batch(n=64):
    """n samples of the data, N(4, 1.25^2), shape (n, 1)."""
    return 4.0 + 1.25 * torch.randn(n, 1)


print("X_V:", tuple(X_V.shape), "· LOGITS_V:", tuple(LOGITS_V.shape), "· MU_V and LOGVAR_V:", tuple(MU_V.shape),
      "· G parameters:", sum(p.numel() for p in make_gan()[0].parameters()))

### CP6.8 — Écrire la loss d'un VAE en PyTorch et la tester sur un batch fourni 📦 ★★ ⏱️ 15 min · 2 points
**Objectif :** écrire de mémoire la loss d'un VAE avec ses deux termes à la bonne échelle.

Avec PyTorch. La cellule des données fournit un batch de 16 images MNIST `X_V` (valeurs dans $[0, 1]$, forme `(16, 1, 28, 28)`), les **logits** qu'un décodeur a rendus pour elles (`LOGITS_V`, même forme) et les deux têtes d'un encodeur à $d = 4$ latents (`MU_V`, `LOGVAR_V`, forme `(16, 4)`) ; tout est en `float64`.

a) `vae_loss(logits, x, mu, log_var, beta=1.0)` rend le triplet `(loss, rec, kl)` de tenseurs à 0 dimension, **par image** (moyennés sur le batch) : `rec`, la cross-entropy binaire entre `x` et `sigmoid(logits)`, **sommée** sur les pixels d'une image ; `kl`, la divergence KL entre $\mathcal{N}(\boldsymbol{\mu}, \operatorname{diag}(e^{\log\boldsymbol{\sigma}^2}))$ et $\mathcal{N}(\mathbf{0}, \mathbf{I})$, sommée sur les latents ; `loss = rec + beta * kl`. Elle marche pour toute forme d'images (`(N, 784)` comme `(N, 1, 28, 28)`) et garde le graphe (le gradient doit pouvoir remonter jusqu'à `logits`, `mu` et `log_var`). **[1,4]**
b) Avec le batch fourni : reporte sur ta feuille $[\text{rec}, \text{kl}]$ (2 décimales). **[0,3]**
c) La loss pour `beta=3` (2 décimales). **[0,3]**

In [ ]:
def vae_loss(logits, x, mu, log_var, beta=1.0):
    """(loss, rec, kl), 0-d tensors per example: rec = BCE summed over the pixels of an image, kl = Gaussian KL summed
    over the latents, both averaged over the batch; loss = rec + beta * kl; CP6.8 a)."""
    n = x.shape[0]
    rec = F.binary_cross_entropy_with_logits(logits, x, reduction="sum") / n        # sum over pixels, mean over images
    kl = 0.5 * (mu ** 2 + log_var.exp() - 1 - log_var).sum() / n                    # sum over latents, mean over images
    return rec + beta * kl, rec, kl


loss_8, rec_8, kl_8 = vae_loss(LOGITS_V, X_V, MU_V, LOGVAR_V)
loss3_8 = vae_loss(LOGITS_V, X_V, MU_V, LOGVAR_V, beta=3.0)[0]
print(f"rec = {rec_8.item():.4f} · kl = {kl_8.item():.4f} · loss = {loss_8.item():.4f} · loss (beta = 3) = {loss3_8.item():.4f}")

In [ ]:
def variant_8(*, rec_mean=False, no_half=False, sigma=False, kl_mean=False, sigmoid_twice=False, no_batch_mean=False,
              beta=1.0):
    """[rec, kl] (or the loss for beta != 1) of CP6.8 written another (wrong) way."""
    n = X_V.shape[0]
    probs = torch.sigmoid(torch.sigmoid(LOGITS_V)) if sigmoid_twice else torch.sigmoid(LOGITS_V)
    bce_all = F.binary_cross_entropy(probs, X_V, reduction="none")
    rec = bce_all.mean() if rec_mean else bce_all.sum() / (1 if no_batch_mean else n)
    var = LOGVAR_V.exp() ** 2 if sigma else LOGVAR_V.exp()                          # log_var read as log(sigma)
    log_v = 2 * LOGVAR_V if sigma else LOGVAR_V
    terms = (MU_V ** 2 + var - 1 - log_v) * (1.0 if no_half else 0.5)
    kl = terms.mean() if kl_mean else terms.sum() / (1 if no_batch_mean else n)
    return [float(rec), float(kl)] if beta == 1.0 else float(rec + beta * kl)


wb.record("CP6.8b", [rec_8.item(), kl_8.item()], decimals=2, mistakes={
    "la reconstruction est SOMMÉE sur les 784 pixels de chaque image, puis moyennée sur les images": variant_8(rec_mean=True),
    "la KL d'une gaussienne contre N(0, 1) a un facteur 1/2 : ½(μ² + σ² − 1 − ln σ²)": variant_8(no_half=True),
    "log_var est le logarithme de la VARIANCE σ² : σ² = exp(log_var)": variant_8(sigma=True),
    "la KL est SOMMÉE sur les 4 latents, puis moyennée sur les images": variant_8(kl_mean=True),
    "les deux termes sont PAR IMAGE : divise les sommes par le nombre d'images": variant_8(no_batch_mean=True),
    "ce sont des logits : binary_cross_entropy_with_logits applique la sigmoïde une seule fois": variant_8(sigmoid_twice=True)})
wb.record("CP6.8c", loss3_8.item(), decimals=2, mistakes={
    "beta multiplie la KL, pas la reconstruction": float(3 * rec_8 + kl_8),
    "la loss vaut rec + beta * kl : avec beta = 3, la KL compte trois fois": loss_8.item(),
    "la KL est SOMMÉE sur les 4 latents, puis moyennée sur les images": variant_8(kl_mean=True, beta=3.0)})

### CP6.9 — Réparer un pas d'entraînement de GAN 🐛 ★ ⏱️ 10 min · 1,5 point
**Objectif :** reconnaître les trois erreurs classiques d'un pas de GAN, et les corriger.

Avec PyTorch. `gan_step_buggy` (cellule suivante) doit faire **un** pas d'entraînement d'un GAN : d'abord le discriminateur (vrais exemples contre faux), puis le générateur (loss non saturante, comme au ch. 27). `make_gan()` rend un générateur, un discriminateur (qui rend un **logit**) et leurs deux optimiseurs ; `real_batch()` tire 64 vrais exemples. Elle contient **trois** lignes fautives ; une erreur peut se déclarer plus loin que sa cause.

a) Écris dans `faulty_9` les lettres des trois lignes **à corriger** (dans n'importe quel ordre). **[0,6]**
b) Écris `gan_step_fixed(G, D, real, opt_d, opt_g)`, la version corrigée : même signature, mêmes sorties, et le même tirage (un seul appel à `torch.randn`, ligne B, avant tout le reste). La cellule d'essai l'enchaîne 1 500 fois (quelques secondes) : la moyenne des exemples générés doit s'approcher de 4. **[0,9]**

In [ ]:
# CP6.9: the step to repair (do not change this cell: copy the function into the next one to fix it)
def gan_step_buggy(G, D, real, opt_d, opt_g):
    """One step of a GAN, the discriminator then the generator; returns (d_loss, g_loss) as floats."""
    ones, zeros = torch.ones(len(real), 1), torch.zeros(len(real), 1)          # A
    fake = G(torch.randn(len(real), 1))                                         # B
    d_loss = bce(D(real), ones) + bce(D(fake), zeros)                           # C
    opt_d.zero_grad()                                                           # D
    d_loss.backward()                                                           # E
    opt_d.step()                                                                # F
    g_loss = bce(D(fake), zeros)                                                # G
    opt_g.zero_grad()                                                           # H
    g_loss.backward()                                                           # I
    opt_d.step()                                                                # J
    return d_loss.item(), g_loss.item()                                         # K

In [ ]:
faulty_9 = "CGJ"


def gan_step_fixed(G, D, real, opt_d, opt_g):
    """gan_step_buggy, fixed (same signature, same returns, one call to torch.randn at line B); CP6.9 b)."""
    ones, zeros = torch.ones(len(real), 1), torch.zeros(len(real), 1)          # A
    fake = G(torch.randn(len(real), 1))                                         # B
    d_loss = bce(D(real), ones) + bce(D(fake.detach()), zeros)                  # C: D's step must not reach G
    opt_d.zero_grad()                                                           # D
    d_loss.backward()                                                           # E
    opt_d.step()                                                                # F
    g_loss = bce(D(fake), ones)                                                 # G: G wants D to answer "real"
    opt_g.zero_grad()                                                           # H
    g_loss.backward()                                                           # I
    opt_g.step()                                                                # J: the generator's optimizer
    return d_loss.item(), g_loss.item()                                         # K


def train_gan(step, n_steps=1500, seed=69):
    """Train make_gan(seed) with `step` on real_batch(); returns (G, the losses of the last 100 steps)."""
    G, D, opt_d, opt_g = make_gan(seed)
    torch.manual_seed(seed + 1)
    losses = [step(G, D, real_batch(), opt_d, opt_g) for _ in range(n_steps)]
    return G, np.mean(losses[-100:], axis=0)


G_9, (d_9, g_9) = train_gan(gan_step_fixed)
with torch.no_grad():
    sample_9 = G_9(torch.randn(2000, 1, generator=torch.Generator().manual_seed(9)))
print(f"generated: mean {sample_9.mean().item():.2f}, std {sample_9.std().item():.2f} (data: 4 and 1.25) · "
      f"last 100 steps: d_loss {d_9:.3f} (2 ln 2 = 1.386), g_loss {g_9:.3f} (ln 2 = 0.693)")

In [ ]:
wb.record("CP6.9a", "CGJ", mistakes={"il manque une ligne fautive : le pas du discriminateur ne doit pas remonter jusqu'au générateur (pense à detach)": "GJ",
                     "il manque une ligne fautive : quel label le générateur veut-il que D donne à ses faux ?": "CJ",
                     "il manque une ligne fautive : quel optimiseur met à jour le générateur ?": "CG",
                     "une ligne de trop : la ligne (A) est correcte ; relis ce qu'elle fait": "ACGJ",
                     "une ligne de trop : la ligne (B) est correcte ; relis ce qu'elle fait": "BCGJ",
                     "une ligne de trop : la ligne (D) est correcte ; relis ce qu'elle fait": "CDGJ",
                     "une ligne de trop : la ligne (E) est correcte ; relis ce qu'elle fait": "CEGJ",
                     "une ligne de trop : la ligne (F) est correcte ; relis ce qu'elle fait": "CFGJ",
                     "une ligne de trop : la ligne (H) est correcte ; relis ce qu'elle fait": "CGHJ",
                     "une ligne de trop : la ligne (I) est correcte ; relis ce qu'elle fait": "CGIJ",
                     "une ligne de trop : la ligne (K) est correcte ; relis ce qu'elle fait": "CGJK",
                     "la ligne (A) est correcte, et il manque une ligne fautive : le pas du discriminateur ne doit pas remonter jusqu'au générateur (pense à detach)": "AGJ",
                     "la ligne (B) est correcte, et il manque une ligne fautive : le pas du discriminateur ne doit pas remonter jusqu'au générateur (pense à detach)": "BGJ",
                     "la ligne (D) est correcte, et il manque une ligne fautive : le pas du discriminateur ne doit pas remonter jusqu'au générateur (pense à detach)": "DGJ",
                     "la ligne (E) est correcte, et il manque une ligne fautive : le pas du discriminateur ne doit pas remonter jusqu'au générateur (pense à detach)": "EGJ",
                     "la ligne (F) est correcte, et il manque une ligne fautive : le pas du discriminateur ne doit pas remonter jusqu'au générateur (pense à detach)": "FGJ",
                     "la ligne (H) est correcte, et il manque une ligne fautive : le pas du discriminateur ne doit pas remonter jusqu'au générateur (pense à detach)": "GHJ",
                     "l'erreur se déclare en (I), mais la ligne à corriger est celle qui laisse le pas de D libérer le graphe du générateur": "GIJ",
                     "la ligne (K) est correcte, et il manque une ligne fautive : le pas du discriminateur ne doit pas remonter jusqu'au générateur (pense à detach)": "GJK",
                     "la ligne (A) est correcte, et il manque une ligne fautive : quel label le générateur veut-il que D donne à ses faux ?": "ACJ",
                     "la ligne (B) est correcte, et il manque une ligne fautive : quel label le générateur veut-il que D donne à ses faux ?": "BCJ",
                     "la ligne (D) est correcte, et il manque une ligne fautive : quel label le générateur veut-il que D donne à ses faux ?": "CDJ",
                     "la ligne (E) est correcte, et il manque une ligne fautive : quel label le générateur veut-il que D donne à ses faux ?": "CEJ",
                     "la ligne (F) est correcte, et il manque une ligne fautive : quel label le générateur veut-il que D donne à ses faux ?": "CFJ",
                     "la ligne (H) est correcte, et il manque une ligne fautive : quel label le générateur veut-il que D donne à ses faux ?": "CHJ",
                     "la ligne (I) est correcte, et il manque une ligne fautive : quel label le générateur veut-il que D donne à ses faux ?": "CIJ",
                     "la ligne (K) est correcte, et il manque une ligne fautive : quel label le générateur veut-il que D donne à ses faux ?": "CJK",
                     "la ligne (A) est correcte, et il manque une ligne fautive : quel optimiseur met à jour le générateur ?": "ACG",
                     "la ligne (B) est correcte, et il manque une ligne fautive : quel optimiseur met à jour le générateur ?": "BCG",
                     "la ligne (D) est correcte, et il manque une ligne fautive : quel optimiseur met à jour le générateur ?": "CDG",
                     "la ligne (E) est correcte, et il manque une ligne fautive : quel optimiseur met à jour le générateur ?": "CEG",
                     "la ligne (F) est correcte, et il manque une ligne fautive : quel optimiseur met à jour le générateur ?": "CFG",
                     "la ligne (H) est correcte, et il manque une ligne fautive : quel optimiseur met à jour le générateur ?": "CGH",
                     "la ligne (I) est correcte, et il manque une ligne fautive : quel optimiseur met à jour le générateur ?": "CGI",
                     "la ligne (K) est correcte, et il manque une ligne fautive : quel optimiseur met à jour le générateur ?": "CGK"})

## Partie B · Les réponses vérifiées après l'examen

Chaque réponse est calculée à partir des nombres du sujet ; les erreurs classiques reçoivent un message qui dit où chercher.

In [ ]:
# The paper questions only use the numbers of their statements (01_examen_sujet.md)
import math

import numpy as np

# CP6.1
RET_1 = 0.9 ** 2 * 1.0

# CP6.2
MU_2, LOGVAR_2, EPS_2 = np.array([0.5, -1.0]), np.array([0.0, math.log(0.25)]), np.array([1.2, -0.4])
SIGMA_2 = np.exp(0.5 * LOGVAR_2)
Z_2 = MU_2 + SIGMA_2 * EPS_2
KL_2 = 0.5 * (MU_2 ** 2 + np.exp(LOGVAR_2) - 1 - LOGVAR_2)
DZ_2 = 0.5 * SIGMA_2[1] * EPS_2[1]                    # dz2 / d(log sigma2^2)
KL_SMALL_2 = 0.5 * (0.0 + math.exp(-10.0) - 1 + 10.0)

# CP6.3
QA_3, QB_3, QC_3 = [0.2, 0.4, 0.1], [0.5, 0.3, 0.8], [0.6, 0.0, 0.0]
ALPHA_3, GAMMA_3, EPS_3 = 0.5, 0.9, 0.2


def update_3(q, target, alpha=ALPHA_3):
    return q + alpha * (target - q)


QL_A_3 = update_3(QA_3[1], 0 + GAMMA_3 * max(QB_3))
QL_B_3 = update_3(QB_3[2], 1.0)                       # C is a winning board: the episode ends
SARSA_A_3 = update_3(QA_3[1], 0 + GAMMA_3 * QB_3[0])
P_GREEDY_3 = 1 - EPS_3 + EPS_3 / 3

# CP6.4
D_EQ_4, G_EQ_4 = 2 * math.log(2), math.log(2)
D_STAR_4 = 0.3 / (0.3 + 0.1)

# CP6.5
F_5 = np.array([[1, 0, 2, 1], [0, 1, 1, 0], [2, 1, 0, 1]], dtype=float)
RAW_5 = F_5 @ F_5.T
G_5 = RAW_5 / F_5.size                                # C * H * W = 3 * 2 * 2

# CP6.10
TABLE_10 = 3 ** 9 * 9 * 4
REACH_10 = 5478 * 9 * 4
BIG_10 = 3 ** 16 * 16 * 4
NET_10 = (48 * 128 + 128) + (128 * 128 + 128) + (128 * 16 + 16)

# CP6.13


def conv_out(n, k, s=1, p=0):
    return (n + 2 * p - k) // s + 1


def adam_13(grads, lr=0.01, b1=0.9, b2=0.999, eps=1e-8, w=1.0):
    m = v = 0.0
    for t, g in enumerate(grads, start=1):
        m, v = b1 * m + (1 - b1) * g, b2 * v + (1 - b2) * g * g
        w -= lr * (m / (1 - b1 ** t)) / (math.sqrt(v / (1 - b2 ** t)) + eps)
    return w

**CP6.1 — Questions flash : dix réponses courtes sur la partie VI**

In [ ]:
wb.record("CP6.1a", 784 / 32, decimals=1, mistakes={"le facteur de compression divise la taille de l'entrée par celle du code": 32 / 784})
wb.record("CP6.1b", False, mistakes={"un autoencodeur ordinaire n'oblige pas ses codes à remplir N(0, I) : un code tiré au hasard tombe souvent dans un trou de l'espace latent": True})
wb.record("CP6.1c", "B", mistakes={"N(μ, σ²) est le posterior approché, celui que rend l'encodeur pour une image ; le prior est fixé d'avance": "A",
                     "le prior du VAE est une gaussienne, pas une loi uniforme": "C"})
wb.record("CP6.1d", RET_1, decimals=2, mistakes={"la récompense arrive au pas 2 : elle est multipliée par γ²": 0.9,
                     "sans γ, c'est le retour non actualisé": 1.0})
wb.record("CP6.1e", "(16, 4)", mistakes={"une ligne par état, une colonne par action": "(4, 16)",
                     "FrozenLake 4 × 4 a 16 états (les cases) et 4 actions (les directions)": "(4, 4)"})
wb.record("CP6.1f", 0.5, decimals=2, mistakes={"à l'équilibre, D ne peut plus distinguer : il répond 1/2 partout, pas 1": 1.0,
                     "1,386 = 2 ln 2 est la loss de D à l'équilibre ; on demande sa SORTIE D(x)": 1.386})
wb.record("CP6.1g", True, mistakes={"l'effondrement de mode, c'est un générateur qui ne couvre qu'une partie des modes (des pics) des données": False})
wb.record("CP6.1h", "(64, 64)", mistakes={"la matrice de Gram croise les cartes deux à deux : une ligne et une colonne par canal": "(196, 196)",
                     "la matrice de Gram ne garde pas les positions : elle est C × C": "(64, 196)"})
wb.record("CP6.1i", 70000, mistakes={"ajoute les 10 000 images de test": 60000})
wb.record("CP6.1j", True, mistakes={"CC BY n'interdit pas l'usage commercial : elle demande de citer la source (c'est NC qui l'interdit)": False})

**CP6.2 — VAE : un tirage reparamétrisé et sa divergence KL**

In [ ]:
wb.record("CP6.2a", Z_2.tolist(), decimals=4, mistakes={"σ = exp(½ log σ²) : log_var n'est pas l'écart-type": (MU_2 + LOGVAR_2 * EPS_2).tolist(),
                     "on multiplie ε par l'écart-type σ, pas par la variance σ²": (MU_2 + np.exp(LOGVAR_2) * EPS_2).tolist()})
wb.record("CP6.2b", KL_2.tolist(), decimals=4, mistakes={"la KL a un facteur ½ : ½(μ² + σ² − 1 − ln σ²)": (2 * KL_2).tolist(),
                     "n'oublie pas le − 1 : la KL de N(0, 1) contre elle-même doit être nulle": (KL_2 + 0.5).tolist(),
                     "ln σ² est log_var lui-même, déjà un logarithme": (0.5 * (MU_2 ** 2 + np.exp(LOGVAR_2) - 1)).tolist()})
wb.record("CP6.2c", DZ_2, decimals=4, mistakes={"z = μ + exp(½ log σ²) ε : la dérivée par rapport à log σ² garde le facteur ½": 2 * DZ_2,
                     "c'est la dérivée par rapport à μ qui vaut 1 ; on demande celle par rapport à log σ²": 1.0})
wb.record("CP6.2d", KL_SMALL_2, decimals=4, mistakes={"le terme − ln σ² vaut + 10 ici : une variance minuscule coûte cher": 0.5 * (math.exp(-10) - 1)})

**CP6.3 — Deux transitions : mise à jour Q-learning puis SARSA**

In [ ]:
wb.record("CP6.3a", QL_A_3, decimals=4, mistakes={"le Q-learning prend le MAX de la ligne de B, pas la valeur de l'action jouée ensuite": SARSA_A_3,
                     "la cible actualise la valeur de B par γ : r + γ max Q(B, ·)": update_3(QA_3[1], max(QB_3)),
                     "on mélange la cible avec l'ancienne valeur dans la proportion α : Q + α (cible − Q)": 0 + GAMMA_3 * max(QB_3)})
wb.record("CP6.3b", QL_B_3, decimals=4, mistakes={"C est un plateau gagnant : l'épisode s'arrête, la cible est r seul (pas d'amorçage sur sa ligne)": update_3(QB_3[2], 1 + GAMMA_3 * max(QC_3)),
                     "on mélange la cible avec l'ancienne valeur dans la proportion α : Q + α (cible − Q)": 1.0})
wb.record("CP6.3c", SARSA_A_3, decimals=4, mistakes={"SARSA prend la valeur de l'action réellement choisie ensuite (a' = 0), pas le max": QL_A_3,
                     "la cible actualise la valeur de B par γ : r + γ Q(B, a')": update_3(QA_3[1], QB_3[0])})
wb.record("CP6.3d", P_GREEDY_3, decimals=4, mistakes={"l'exploration tire une action au hasard parmi les TROIS, la meilleure comprise : elle aussi peut tomber sur l'action 2": 1 - EPS_3,
                     "c'est la probabilité d'une action non gloutonne donnée : on demande celle de l'action gloutonne": EPS_3 / 3})

**CP6.4 — Discriminateur optimal et losses à l'équilibre**

In [ ]:
wb.record("CP6.4b", [D_EQ_4, G_EQ_4], decimals=3, mistakes={"la loss de D additionne deux cross-entropies, chacune vaut ln 2 à l'équilibre": [G_EQ_4, G_EQ_4],
                     "la loss non saturante de G est − ln D(G(z)), un seul terme : ln 2": [D_EQ_4, D_EQ_4],
                     "c'est la sortie de D à l'équilibre ; on demande les losses": [0.5, 0.5]})
wb.record("CP6.4c", D_STAR_4, decimals=4, mistakes={"D* = p_data / (p_data + p_g), pas p_g / (p_data + p_g)": 0.1 / 0.4,
                     "D* divise par la SOMME des deux densités": 0.3 / 0.1})

**CP6.5 — Matrice de Gram de trois cartes d'activation**

In [ ]:
wb.record("CP6.5a", RAW_5.tolist(), decimals=1, mistakes={"F F^T : produit des cartes mises à plat ligne par ligne, une ligne et une colonne par canal (C × C)": (F_5.T @ F_5).tolist()})
wb.record("CP6.5b", [G_5[0, 0], G_5[0, 2]], decimals=4, mistakes={"la fiche divise par C·H·W = 12, pas par H·W seulement": [RAW_5[0, 0] / 4, RAW_5[0, 2] / 4],
                     "la fiche divise par C·H·W = 12, pas par C seulement": [RAW_5[0, 0] / 3, RAW_5[0, 2] / 3],
                     "la matrice de Gram normalisée divise F F^T par C·H·W": [RAW_5[0, 0], RAW_5[0, 2]]})

**CP6.6 — Prédire : γ → 0 puis ε = 0 pour un agent Q-learning sur Flippers**

In [ ]:
wb.record("CP6.6a", "C", mistakes={"avec γ = 0, une case Q ne voit que la récompense immédiate : loin de la victoire, toutes les valeurs restent nulles": "A",
                     "avec γ = 0, seuls les plateaux à UN coup de la victoire apprennent quelque chose : regarde quelle part ils représentent": "B"})
wb.record("CP6.6b", "A", mistakes={"tant que la table est nulle, les égalités se départagent AU HASARD pendant l'entraînement : l'agent explore quand même": "B",
                     "tant que la table est nulle, les égalités se départagent AU HASARD pendant l'entraînement : l'agent explore quand même ; γ = 0,9 propage ensuite les victoires": "C"})

**CP6.7 — Trois courbes, trois diagnostics : VAE, GAN, apprentissage par renforcement**

In [ ]:
wb.record("CP6.7a", "A", mistakes={"regarde la KL (axe de droite) : elle reste collée à 0 et la reconstruction plafonne vers 208": "B",
                     "les courbes ① sont celles d'un VAE (une reconstruction et une KL), pas d'un GAN": "C",
                     "un VAE n'a ni discriminateur ni générateur à mettre en équilibre : lis la KL": "D",
                     "les courbes ① sont celles d'un VAE, pas d'un agent": "E",
                     "un VAE n'a pas de récompense : lis la KL, sur l'axe de droite": "F"})
wb.record("CP6.7b", "C", mistakes={"les courbes ② sont celles d'un GAN (une loss de D, une loss de G), pas d'un VAE": "A",
                     "un GAN n'a ni reconstruction ni KL : compare les deux losses": "B",
                     "à l'équilibre, la loss de D vaudrait 2 ln 2 ≈ 1,39 ; ici elle est presque nulle et celle de G dépasse 6": "D",
                     "les courbes ② sont celles d'un GAN, pas d'un agent": "E",
                     "un GAN n'a pas de récompense : compare les deux losses": "F"})
wb.record("CP6.7c", "F", mistakes={"les courbes ③ sont celles d'un agent (un retour par épisode), pas d'un VAE": "A",
                     "un agent n'a ni reconstruction ni KL : lis le niveau du retour": "B",
                     "les courbes ③ sont celles d'un agent, pas d'un GAN": "C",
                     "un agent n'a pas de discriminateur : lis le niveau du retour": "D",
                     "le retour d'un épisode vaut 1 pour une victoire : après 3 000 épisodes, il reste sous 0,1, loin d'une bonne politique": "E"})

**CP6.10 — Table Q ou réseau : estimer la mémoire nécessaire**

In [ ]:
wb.record("CP6.10a", TABLE_10 / 1e6, decimals=2, mistakes={"un float32 occupe 4 octets": 3 ** 9 * 9 / 1e6,
                     "une ligne par codage de plateau (3⁹), une colonne par case (9)": 3 ** 9 * 4 / 1e6,
                     "chaque case a 3 états possibles : 3⁹ codages, pas 2⁹": 2 ** 9 * 9 * 4 / 1e6})
wb.record("CP6.10b", REACH_10 / 1e6, decimals=2, mistakes={"un float32 occupe 4 octets": 5478 * 9 / 1e6})
wb.record("CP6.10c", BIG_10 / 1e9, decimals=3, mistakes={"le morpion 4 × 4 a 16 cases, donc 16 actions par ligne": 3 ** 16 * 9 * 4 / 1e9,
                     "un float32 occupe 4 octets": 3 ** 16 * 16 / 1e9})
wb.record("CP6.10d", NET_10, mistakes={"compte aussi les biais, un par neurone": NET_10 - (128 + 128 + 16),
                     "l'entrée a 16 cases × 3 états (codage one-hot), soit 48 valeurs": (16 * 128 + 128) + (128 * 128 + 128) + (128 * 16 + 16)})

**CP6.13 — Retour sur les parties IV et V : convolution et pas d'Adam**

In [ ]:
wb.record("CP6.13a", "(1, 32, 32, 32)", mistakes={"le padding de 2 s'ajoute des deux côtés : (64 + 4 − 5) / 2 + 1, arrondi vers le bas": "(1, 32, 30, 30)",
                     "le stride de 2 divise la taille par deux": "(1, 32, 64, 64)",
                     "la sortie a autant de canaux que de filtres": "(1, 3, 32, 32)"})
wb.record("CP6.13b", 32 * 3 * 5 * 5 + 32, mistakes={"un filtre couvre les 3 canaux d'entrée": 32 * 5 * 5 + 32,
                     "compte aussi les biais, un par filtre": 32 * 3 * 5 * 5})
wb.record("CP6.13c", "(1, 16, 64, 64)", mistakes={"c'est la formule d'une convolution ordinaire : la convolution transposée agrandit": "(1, 16, 16, 16)",
                     "le padding d'une convolution transposée retire 2p à la taille : (32 − 1) × 2 − 2 + 4": "(1, 16, 66, 66)"})
wb.record("CP6.13d", adam_13([0.5]), decimals=4, mistakes={"la correction du biais divise m₁ par 1 − β₁ : le premier pas vaut environ lr, quelle que soit la taille du gradient": 1.0 - 0.01 * 0.05 / (math.sqrt(0.00025) + 1e-8),
                     "c'est le pas de la descente de gradient simple, w − lr g ; Adam normalise par la racine de v": 1.0 - 0.01 * 0.5})
wb.record("CP6.13e", adam_13([0.5, -0.1]), decimals=4, mistakes={"au 2ᵉ pas, la correction du biais divise par 1 − β₁² et 1 − β₂², pas par 1 − β₁ et 1 − β₂": adam_13([0.5]) - 0.01 * (0.035 / 0.1) / (math.sqrt(0.00025975 / 0.001) + 1e-8),
                     "Adam garde la mémoire de ses moyennes : le 2ᵉ pas n'est pas un nouveau premier pas de taille lr": 0.99 + 0.01})